# All of Us HF Feature Extraction and Harmonization with JHU Metadata

This notebook extracts patient-level features for a previously defined All of Us (AoU) heart failure (HF) cohort and maps them to covariate identifiers from Johns Hopkins University (JHU) feature metadata. Each patient's first recorded HF diagnosis serves as the index date. The resulting feature matrix supports downstream transfer of the JHU model to the AoU cohort.

## Required inputs

- `meta_HFCohort_deid.csv`: JHU feature metadata containing covariate IDs, names, concept IDs, and analysis IDs.
- `unique_patient_ids_with_hf_date_deaths.csv`: Previously generated cohort file containing patient IDs, first HF diagnosis dates, and age at diagnosis.
- An authenticated connection to the AoU OMOP dataset through the `allofus` R package.

## Feature extraction

- **Age:** Approximate age at the index date, calculated using January 1 of the recorded birth year.
- **Conditions:** Binary indicators for matching condition eras starting on or before the index date, with no lower time bound (analysis ID 201).
- **Measurements:** Latest nonmissing numeric measurement per patient, concept, and unit from 365 days before through 7 days after the index date (analysis ID 706). Selected unit labels and height/weight values are harmonized with JHU metadata.
- **Derived eGFR:** Calculated using the female, race-free CKD-EPI 2021 equation, creatinine values, and age at diagnosis.
- **Drugs:** Binary indicators for matching drug eras overlapping the interval from 730 days before through the index date (analysis ID 402).
- **Procedures and observations:** Binary indicators for matching records on or before the index date, with no lower time bound (analysis IDs 501 and 801).

## Output

Domain-specific features are joined into a wide patient-level matrix. Missing binary indicators are filled with zero, while missing continuous values remain `NA`. Diagnostic checks summarize feature coverage, unmatched concepts, matrix dimensions, and duplicate patient IDs.

The final matrix is saved as `feature_matrix_wide_deaths.csv`, with `person_id` identifying patients and feature columns named by JHU covariate ID.

In [ ]:
# load libraries
library(allofus)
library(dplyr)
library(tidyverse)
con <- aou_connect()
library(bit64)
library(stringr)

In [ ]:
# load metadata from jhu ehr side
jhu_raw  <- read.csv('data/meta_HFCohort_deid.csv', header = TRUE)
jhu_cols <- as.numeric(jhu_raw[[1]])

jhu_df <- data.frame(
  covariate_id   = jhu_raw$covariateId,
  covariate_name = jhu_raw$covariateName,
  concept_id     = jhu_raw$conceptId,
  analysis_id    = jhu_raw$analysisId
)
dup_concept_ids <- jhu_df %>%
  count(concept_id, sort = TRUE) %>%
  filter(n > 1) %>%
  pull(concept_id)

jhu_df %>%
  filter(concept_id %in% dup_concept_ids) %>%
  arrange(concept_id, analysis_id)

In [ ]:
table(jhu_df$analysis_id)

# Cohort Definition

In [ ]:
hf_dates_initial <- read_csv("unique_patient_ids_with_hf_date_deaths.csv")

# Only HF patient IDs
cohort_ids <- unique(as.integer(hf_dates_initial$PERSON_ID))
cohort_ids <- cohort_ids[!is.na(cohort_ids)]

person_tbl <- tbl(con, "person")

cohort <- person_tbl %>%
  filter(person_id %in% cohort_ids) %>%
  collect()



cohort_with_dates <- cohort %>%
  mutate(person_id = as.character(person_id)) %>%
  left_join(
    hf_dates_initial %>%
      mutate(hf_person_id = as.character(PERSON_ID)) %>%
      select(hf_person_id, first_hf_diagnosis_date),
    by = c("person_id" = "hf_person_id")
  )

hf_dates <- cohort_with_dates %>%
  select(person_id, first_hf_diagnosis_date) %>%
  distinct()

cohort_person_ids <- hf_dates$person_id
cohort_ids_int <- as.integer(hf_dates$person_id)

In [ ]:
cat("cohort_ids:", length(cohort_ids), "\n")          
cat("cohort rows:", nrow(cohort), "\n")              
cat("hf_dates rows:", nrow(hf_dates), "\n")            
cat("Patients without HF date:", 
    sum(is.na(hf_dates$first_hf_diagnosis_date)), "\n") 

# Age and Age Group

In [ ]:
# Get JHU age covariate id (analysis_id == 2 only)
jhu_age_id        <- jhu_df %>% filter(analysis_id == 2) %>% pull(covariate_id)
jhu_covariate_ids <- jhu_df$covariate_id

demo_base <- cohort %>%
  mutate(person_id = as.character(person_id)) %>%
  left_join(
    hf_dates %>% rename(index_date = first_hf_diagnosis_date),
    by = "person_id"
  ) %>%
  mutate(
    age_at_index = as.integer(
      floor(as.numeric(difftime(as.Date(index_date),
                                as.Date(paste0(year_of_birth, "-01-01")),
                                units = "days")) / 365.25)
    )
  )

demo_age <- demo_base %>%
  select(person_id, age_at_index) %>%
  filter(!is.na(age_at_index)) %>%
  mutate(
    covariate_id = jhu_age_id,
    value        = age_at_index
  ) %>%
  select(person_id, covariate_id, value) %>%
  filter(covariate_id %in% jhu_covariate_ids)

cat("demo_age rows:", nrow(demo_age),
    "| unique patients:", n_distinct(demo_age$person_id), "\n")

demo_age %>%
  count(covariate_id) %>%
  left_join(jhu_df %>% select(covariate_id, covariate_name), by = "covariate_id") %>%
  arrange(covariate_id)

In [ ]:
age_continuous_wide <- demo_age %>%
  pivot_wider(
    id_cols     = person_id,
    names_from  = covariate_id,
    values_from = value,
    values_fill = NA
  )

# Conditions

In [ ]:
## final version (includes only analysis ID 201, ever prior)
cond_era_tbl <- tbl(con, "condition_era")

apply_time_window_overlap <- function(df, start_col, end_col, window_days = NULL) {
  df %>%
    mutate(person_id = as.character(person_id)) %>%
    left_join(
      hf_dates %>%
        mutate(person_id = as.character(person_id)) %>%
        rename(index_date = first_hf_diagnosis_date),
      by = "person_id"
    ) %>%
    mutate(
      index_date   = as.Date(index_date),
      window_end   = index_date,
      window_start = if (is.null(window_days)) as.Date(-Inf) else index_date - window_days
    ) %>%
    filter(
      .data[[start_col]] <= window_end,
      .data[[end_col]]   >= window_start
    ) %>%
    select(-index_date, -window_start, -window_end)
}

# JHU concept_ids for analysis_id 201, computed once
jhu_cond_era_concept_ids <- jhu_df %>%
  filter(analysis_id == 201) %>%
  pull(concept_id) %>%
  as.numeric()

cond_era_all <- cond_era_tbl %>%
  filter(person_id %in% cohort_ids_int) %>%
  collect() %>%
  apply_time_window_overlap("condition_era_start_date", "condition_era_end_date", window_days = NULL) %>%
  select(person_id, condition_concept_id) %>%
  distinct() %>%
  mutate(
    condition_concept_id = as.numeric(condition_concept_id),
    analysis_id  = 201L,
    covariate_id = condition_concept_id * 1000 + 201L
  ) %>%
  rename(concept_id = condition_concept_id) %>%
  filter(concept_id %in% jhu_cond_era_concept_ids)

cat("cond_era_all rows:", nrow(cond_era_all),
    "| unique covariates:", n_distinct(cond_era_all$covariate_id),
    "| unique patients:", n_distinct(cond_era_all$person_id), "\n")

cond_era_all_wide <- cond_era_all %>%
  mutate(value = 1L) %>%
  pivot_wider(
    id_cols     = person_id,
    names_from  = covariate_id,
    values_from = value,
    values_fill = 0L
  )

cat("JHU condition era concepts:", length(jhu_cond_era_concept_ids), "\n")
cat("AoU condition era concepts:", n_distinct(cond_era_all$concept_id), "\n")
cat("Overlap:", sum(unique(cond_era_all$concept_id) %in% jhu_cond_era_concept_ids), "\n")

## Missing Data Analysis - Conditions

In [ ]:
# Missing concept_ids: in JHU's 201 list but not matched in cond_era_all
missing_era_concept_ids <- setdiff(jhu_cond_era_concept_ids, unique(cond_era_all$concept_id))
cat("Missing in AoU (era, 201):", length(missing_era_concept_ids), "\n")

In [ ]:
# 1. Do these concepts exist in condition_era at all (raw, no person/time filters)?
era_raw_check <- cond_era_tbl %>%
  filter(condition_concept_id %in% missing_era_concept_ids) %>%
  count(condition_concept_id) %>%
  collect()
cat("Concepts existing in AoU condition_era raw (no filter):", nrow(era_raw_check), "/", length(missing_era_concept_ids), "\n")

# 2. After cohort filter (restrict to your cohort's person_ids)
era_cohort_check <- cond_era_tbl %>%
  filter(person_id %in% cohort_ids_int,
         condition_concept_id %in% missing_era_concept_ids) %>%
  count(condition_concept_id) %>%
  collect()
cat("After cohort filter:", nrow(era_cohort_check), "\n")

# 3. After time window overlap 
era_time_check <- cond_era_tbl %>%
  filter(person_id %in% cohort_ids_int,
         condition_concept_id %in% missing_era_concept_ids) %>%
  collect() %>%
  mutate(person_id = as.character(person_id)) %>%
  apply_time_window_overlap("condition_era_start_date", "condition_era_end_date", window_days = 730) %>%
  distinct(condition_concept_id)
cat("After time window:", nrow(era_time_check), "\n")

# Measurements

In [ ]:
unit_aliases <- tribble(
  ~unit_concept_name_raw,      ~unit_concept_name_jhu,
  "IU/L",                       "international unit per liter"
)

# Numeric unit conversions -- for concepts where AoU records a DIFFERENT
# physical unit than JHU (not just a different text label for the same
# unit, which is what unit_aliases handles). Each row: which concept, which
# raw AoU unit, what JHU's unit is called, and the multiplicative factor to
# convert AoU's raw value into JHU's unit.
unit_conversions <- tribble(
  ~concept_id, ~unit_concept_name_raw, ~unit_concept_name_jhu,          ~factor,
  3036277,     "centimeter",           "inch (US)",                      1 / 2.54,
  3025315,     "kilogram",             "pound force (avoirdupois)",      2.20462,
)

analysis_id_706 <- 706L

jhu_meas_706_concepts <- jhu_df %>%
  filter(analysis_id == 706) %>%
  pull(concept_id) %>%
  as.numeric()

meas_tbl    <- tbl(con, "measurement")
concept_tbl <- tbl(con, "concept")

apply_window_meas <- function(df, date_col, start_days, end_days) {
  df %>%
    mutate(person_id = as.character(person_id)) %>%
    left_join(
      hf_dates %>%
        mutate(person_id = as.character(person_id)) %>%
        rename(index_date = first_hf_diagnosis_date),
      by = "person_id"
    ) %>%
    mutate(index_date = as.Date(index_date)) %>%
    filter(
      .data[[date_col]] >= index_date + start_days,
      .data[[date_col]] <= index_date + end_days
    ) %>%
    select(-index_date)
}

meas_706_concepts <- unique(c(jhu_meas_706_concepts, 3036277, 3025315))

build_meas_features <- function(concept_ids, analysis_id_val, start_days, end_days) {

  meas_raw <- meas_tbl %>%
    filter(person_id %in% cohort_ids_int,
           measurement_concept_id %in% concept_ids) %>%
    select(person_id, measurement_concept_id, measurement_date,
           value_as_number, unit_concept_id) %>%
    collect() %>%
    mutate(
      measurement_concept_id = as.numeric(measurement_concept_id),
      unit_concept_id        = as.numeric(unit_concept_id)
    ) %>%
    apply_window_meas("measurement_date", start_days, end_days)

  # chronologically LAST measurement per person/concept/unit -- matches
  # FeatureExtraction's actual useMeasurementValueLongTerm behavior
  meas_agg <- meas_raw %>%
    filter(!is.na(value_as_number)) %>%
    group_by(person_id, measurement_concept_id, unit_concept_id) %>%
    slice_max(measurement_date, n = 1, with_ties = FALSE) %>%
    ungroup() %>%
    select(person_id, measurement_concept_id, unit_concept_id, value_as_number)

  meas_concept_ids <- unique(meas_agg$measurement_concept_id)
  unit_concept_ids <- unique(meas_agg$unit_concept_id[!is.na(meas_agg$unit_concept_id)])

  meas_names <- concept_tbl %>%
    filter(concept_id %in% !!meas_concept_ids) %>%
    select(concept_id, concept_name) %>%
    collect() %>%
    mutate(concept_id = as.numeric(concept_id)) %>%
    rename(measurement_concept_id = concept_id, measurement_concept_name = concept_name)

  unit_names <- concept_tbl %>%
    filter(concept_id %in% !!unit_concept_ids) %>%
    select(concept_id, concept_name) %>%
    collect() %>%
    mutate(concept_id = as.numeric(concept_id)) %>%
    rename(unit_concept_id = concept_id, unit_concept_name = concept_name)

  meas_agg <- meas_agg %>%
    left_join(meas_names, by = "measurement_concept_id") %>%
    left_join(unit_names, by = "unit_concept_id") %>%
    left_join(unit_aliases, by = c("unit_concept_name" = "unit_concept_name_raw")) %>%
    # numeric unit conversion, applied only to the concept/unit combinations
    # listed in unit_conversions -- everything else passes through untouched
    left_join(
      unit_conversions,
      by = c("measurement_concept_id" = "concept_id", "unit_concept_name" = "unit_concept_name_raw")
    ) %>%
    mutate(
      value_as_number = if_else(!is.na(factor), value_as_number * factor, value_as_number),
      unit_concept_name_matched = coalesce(unit_concept_name_jhu.y, unit_concept_name_jhu.x, unit_concept_name),
      covariate_name = sprintf(
        "measurement value during day %d through %d days relative to index: %s (%s)",
        start_days, end_days, measurement_concept_name, unit_concept_name_matched
      ),
      analysis_id = analysis_id_val,
      concept_id  = measurement_concept_id
    )

  meas_agg %>%
    left_join(
      jhu_df %>% filter(analysis_id == analysis_id_val) %>% select(covariate_id, covariate_name),
      by = "covariate_name"
    ) %>%
    select(person_id, concept_id, unit_concept_id, unit_concept_name,
           unit_concept_name_matched, covariate_name, covariate_id,
           analysis_id, value_as_number)
}

meas_706 <- build_meas_features(meas_706_concepts, analysis_id_706, -365, 7)

cat("meas_706 rows:", nrow(meas_706),
    "| unique covariates:", n_distinct(meas_706$covariate_id[!is.na(meas_706$covariate_id)]),
    "| unique patients:", n_distinct(meas_706$person_id), "\n")

# verify height/weight specifically landed on JHU's covariate_ids
meas_706 %>%
  filter(concept_id %in% c(3036277, 3025315)) %>%
  distinct(concept_id, unit_concept_name, unit_concept_name_matched, covariate_name, covariate_id) %>%
  print()

In [ ]:
meas_706_wide <- meas_706 %>%
  filter(!is.na(covariate_id)) %>%
  select(person_id, covariate_id, value_as_number) %>%
  distinct(person_id, covariate_id, .keep_all = TRUE) %>%
  pivot_wider(id_cols = person_id, names_from = covariate_id,
              values_from = value_as_number, values_fill = NA)

cat("meas_706_wide:", nrow(meas_706_wide), "patients x",
    ncol(meas_706_wide) - 1, "measurements\n")

In [ ]:
# CKD-EPI 2021 eGFR (race-free, female)
compute_egfr_ckdepi_2021_female <- function(scr_mgdl, age_years) {
  kappa <- 0.7; alpha <- -0.241
  ratio <- scr_mgdl / kappa
  142 * pmin(ratio, 1)^alpha * pmax(ratio, 1)^(-1.200) * (0.9938^age_years) * 1.012
}
creat_concept <- 3016723

cohort_index <- hf_dates_initial %>%
 transmute(person_id = as.character(PERSON_ID),
            age_at_index = as.numeric(age_at_diagnosis))



egfr_computed <- meas_706 %>%
  filter(concept_id == creat_concept, !is.na(value_as_number)) %>%
  distinct(person_id, .keep_all = TRUE) %>%
  inner_join(cohort_index, by = "person_id") %>%
  mutate(egfr_ckdepi_2021 = compute_egfr_ckdepi_2021_female(value_as_number, age_at_index))

cat("\nPatients with computed eGFR:", nrow(egfr_computed), "\n")
print(summary(egfr_computed$egfr_ckdepi_2021))

jhu_egfr <- jhu_df %>%
  filter(analysis_id == 706, grepl("glomerular filtration", covariate_name, ignore.case = TRUE)) %>%
  select(covariate_id, covariate_name)
print(jhu_egfr)
jhu_egfr_id <- as.character(jhu_egfr$covariate_id[1])

egfr_col <- egfr_computed %>%
  transmute(person_id, !!jhu_egfr_id := egfr_ckdepi_2021)
meas_706_wide <- meas_706_wide %>% left_join(egfr_col, by = "person_id")

# Drugs

In [ ]:
## final version (only include analysis ID 402 - drug era long term, -730 to 0 days)
drug_era_tbl <- tbl(con, "drug_era")

apply_time_window_overlap <- function(df, start_col, end_col, window_days = 730) {
  df %>%
    mutate(person_id = as.character(person_id)) %>%
    left_join(
      hf_dates %>%
        mutate(person_id = as.character(person_id)) %>%
        rename(index_date = first_hf_diagnosis_date),
      by = "person_id"
    ) %>%
    mutate(
      index_date   = as.Date(index_date),
      window_start = index_date - window_days,
      window_end   = index_date
    ) %>%
    filter(
      .data[[start_col]] <= window_end,
      .data[[end_col]]   >= window_start
    ) %>%
    select(-index_date, -window_start, -window_end)
}

# JHU concept_ids for analysis_id 402, computed once
jhu_drug_era_concept_ids <- jhu_df %>%
  filter(analysis_id == 402) %>%
  pull(concept_id) %>%
  as.numeric()

drug_era_all <- drug_era_tbl %>%
  filter(person_id %in% cohort_ids_int) %>%
  collect() %>%
  apply_time_window_overlap("drug_era_start_date", "drug_era_end_date", window_days = 730) %>%
  select(person_id, drug_concept_id) %>%
  distinct() %>%
  mutate(
    drug_concept_id = as.numeric(drug_concept_id),  # integer64 -> numeric fix
    analysis_id  = 402L,
    covariate_id = drug_concept_id * 1000 + 402L
  ) %>%
  rename(concept_id = drug_concept_id) %>%
  filter(concept_id %in% jhu_drug_era_concept_ids)

cat("drug_era_all rows:", nrow(drug_era_all),
    "| unique covariates:", n_distinct(drug_era_all$covariate_id),
    "| unique patients:", n_distinct(drug_era_all$person_id), "\n")

drug_era_all_wide <- drug_era_all %>%
  mutate(value = 1L) %>%
  pivot_wider(
    id_cols     = person_id,
    names_from  = covariate_id,
    values_from = value,
    values_fill = 0L
  )

cat("JHU drug era concepts:", length(jhu_drug_era_concept_ids), "\n")
cat("AoU drug era concepts:", n_distinct(drug_era_all$concept_id), "\n")
cat("Overlap:", sum(unique(drug_era_all$concept_id) %in% jhu_drug_era_concept_ids), "\n")

print(dim(drug_era_all_wide))

## Missing Data

In [ ]:
# 1. Missing in AoU (concept-based, matching drug_era_all's filtering logic)
missing_drug_era_concept_ids <- setdiff(jhu_drug_era_concept_ids, unique(drug_era_all$concept_id))
cat("Missing in AoU:", length(missing_drug_era_concept_ids), "\n")

# 2. Exist in AoU raw
drug_era_raw_check <- drug_era_tbl %>%
  filter(drug_concept_id %in% missing_drug_era_concept_ids) %>%
  count(drug_concept_id) %>%
  collect()
cat("Concepts existing in AoU raw (no filter):", nrow(drug_era_raw_check), "/", length(missing_drug_era_concept_ids), "\n")

# 3. After cohort filter
drug_era_cohort_check <- drug_era_tbl %>%
  filter(person_id %in% cohort_ids_int,
         drug_concept_id %in% missing_drug_era_concept_ids) %>%
  count(drug_concept_id) %>%
  collect()
cat("After cohort filter:", nrow(drug_era_cohort_check), "\n")

# 4. After time window (era overlap, -730 to 0)
drug_era_time_check <- drug_era_tbl %>%
  filter(person_id %in% cohort_ids_int,
         drug_concept_id %in% missing_drug_era_concept_ids) %>%
  collect() %>%
  mutate(person_id = as.character(person_id)) %>%
  apply_time_window_overlap("drug_era_start_date", "drug_era_end_date", window_days = 730) %>%
  distinct(drug_concept_id)
cat("After time window:", nrow(drug_era_time_check), "\n")

# Procedures

In [ ]:
## final version (includes only analysis ID 501, ANY TIME prior to index date)
proc_occ_tbl <- tbl(con, "procedure_occurrence")

apply_time_window <- function(df, date_col, window_days = NULL) {
  df %>%
    mutate(person_id = as.character(person_id)) %>%
    left_join(
      hf_dates %>%
        mutate(person_id = as.character(person_id)) %>%
        rename(index_date = first_hf_diagnosis_date),
      by = "person_id"
    ) %>%
    mutate(index_date = as.Date(index_date)) %>%
    filter(
      if (!is.null(window_days)) .data[[date_col]] >= index_date - window_days else TRUE,
      .data[[date_col]] <= index_date
    ) %>%
    select(-index_date)
}

jhu_proc_concept_ids <- jhu_df %>%
  filter(analysis_id == 501) %>%
  pull(concept_id) %>%
  as.numeric()

proc_all <- proc_occ_tbl %>%
  filter(person_id %in% cohort_ids_int) %>%
  collect() %>%
  apply_time_window("procedure_date", window_days = NULL) %>%
  select(person_id, procedure_concept_id) %>%
  distinct() %>%
  mutate(
    procedure_concept_id = as.numeric(procedure_concept_id),
    analysis_id  = 501L,
    covariate_id = procedure_concept_id * 1000 + 501L
  ) %>%
  rename(concept_id = procedure_concept_id) %>%
  filter(concept_id %in% jhu_proc_concept_ids) # filter by concept_id, not covariate_id

cat("proc_all rows:", nrow(proc_all),
    "| unique covariates:", n_distinct(proc_all$covariate_id),
    "| unique patients:", n_distinct(proc_all$person_id), "\n")

proc_all_wide <- proc_all %>%
  mutate(value = 1L) %>%
  pivot_wider(
    id_cols     = person_id,
    names_from  = covariate_id,
    values_from = value,
    values_fill = 0L
  )

jhu_proc_ids <- jhu_df %>% filter(analysis_id == 501) %>% pull(covariate_id)
cat("JHU procedure covariates:", length(jhu_proc_ids), "\n")
cat("AoU procedure covariates:", n_distinct(proc_all$covariate_id), "\n")
cat("Overlap:", sum(unique(proc_all$concept_id) %in% jhu_proc_concept_ids), "\n")

# Missing Data Analysis - Procedures

In [ ]:
# 1. Missing in AoU (concept-based, matching proc_all's actual filtering logic)
missing_proc_concept_ids <- setdiff(jhu_proc_concept_ids, unique(proc_all$concept_id))
cat("Missing in AoU:", length(missing_proc_concept_ids), "\n")

# 2. Exist in AoU raw
proc_raw_check <- proc_occ_tbl %>%
  filter(procedure_concept_id %in% missing_proc_concept_ids) %>%
  count(procedure_concept_id) %>%
  collect()
cat("Concepts existing in AoU raw (no filter):", nrow(proc_raw_check), "/", length(missing_proc_concept_ids), "\n")

# 3. After cohort filter
proc_cohort_check <- proc_occ_tbl %>%
  filter(person_id %in% cohort_ids_int,
         procedure_concept_id %in% missing_proc_concept_ids) %>%
  count(procedure_concept_id) %>%
  collect()
cat("After cohort filter:", nrow(proc_cohort_check), "\n")

# 4. After time window (-730 to 0, matching proc_all)
proc_time_check <- proc_occ_tbl %>%
  filter(person_id %in% cohort_ids_int,
         procedure_concept_id %in% missing_proc_concept_ids) %>%
  collect() %>%
  mutate(person_id = as.character(person_id)) %>%
  apply_time_window("procedure_date", window_days = 730) %>%
  distinct(procedure_concept_id)
cat("After time window:", nrow(proc_time_check), "\n")

# Observation

In [ ]:
## final version (analysis id 801 - observation, ANY TIME prior to index date)
obs_tbl <- tbl(con, "observation")

apply_window_obs <- function(df, date_col, start_days = NULL, end_days) {
  df %>%
    mutate(person_id = as.character(person_id)) %>%
    left_join(hf_dates %>% rename(index_date = first_hf_diagnosis_date), by = "person_id") %>%
    mutate(index_date = as.Date(index_date)) %>%
    filter(
      if (!is.null(start_days)) .data[[date_col]] >= index_date + start_days else TRUE,
      .data[[date_col]] <= index_date + end_days
    ) %>%
    select(-index_date)
}

jhu_obs_801_concept_ids <- jhu_df %>%
  filter(analysis_id == 801) %>%
  pull(concept_id) %>%
  as.numeric()

build_obs_window <- function(analysis_id_val, start_days = NULL, end_days) {
  obs_tbl %>%
    filter(person_id %in% cohort_ids_int) %>%
    select(person_id, observation_concept_id, observation_date) %>%
    collect() %>%
    apply_window_obs("observation_date", start_days, end_days) %>%
    select(person_id, observation_concept_id) %>%
    distinct() %>%
    mutate(
      observation_concept_id = as.numeric(observation_concept_id),  # integer64 -> numeric fix
      analysis_id  = analysis_id_val,
      covariate_id = observation_concept_id * 1000 + analysis_id_val
    ) %>%
    rename(concept_id = observation_concept_id) %>%
    filter(concept_id %in% jhu_obs_801_concept_ids)  # filter by concept_id, not covariate_id
}

obs_801 <- build_obs_window(801L, start_days = NULL, end_days = 0)

obs_801_wide <- obs_801 %>%
  mutate(value = 1L) %>%
  pivot_wider(
    id_cols     = person_id,
    names_from  = covariate_id,
    values_from = value,
    values_fill = 0L
  )

cat("obs_801 rows:", nrow(obs_801),
    "| unique covariates:", n_distinct(obs_801$covariate_id),
    "| unique patients:", n_distinct(obs_801$person_id), "\n")
cat("JHU obs 801 concepts:", length(jhu_obs_801_concept_ids), "\n")
cat("AoU obs 801 concepts:", n_distinct(obs_801$concept_id), "\n")
cat("Overlap:", sum(unique(obs_801$concept_id) %in% jhu_obs_801_concept_ids), "\n")

## Missing Data

In [ ]:
# 1. Missing in AoU (concept-based, matching obs_801's filtering logic)
missing_obs_concept_ids <- setdiff(jhu_obs_801_concept_ids, unique(obs_801$concept_id))
cat("Missing in AoU:", length(missing_obs_concept_ids), "\n")

# 2. Exist in AoU raw
obs_raw_check <- obs_tbl %>%
  filter(observation_concept_id %in% missing_obs_concept_ids) %>%
  count(observation_concept_id) %>%
  collect()
cat("Concepts existing in AoU raw (no filter):", nrow(obs_raw_check), "/", length(missing_obs_concept_ids), "\n")

# 3. After cohort filter
obs_cohort_check <- obs_tbl %>%
  filter(person_id %in% cohort_ids_int,
         observation_concept_id %in% missing_obs_concept_ids) %>%
  count(observation_concept_id) %>%
  collect()
cat("After cohort filter:", nrow(obs_cohort_check), "\n")

# 4. After time window (any time prior to index, matching obs_801)
obs_time_check <- obs_tbl %>%
  filter(person_id %in% cohort_ids_int,
         observation_concept_id %in% missing_obs_concept_ids) %>%
  collect() %>%
  mutate(person_id = as.character(person_id)) %>%
  apply_window_obs("observation_date", NULL, 0)
cat("After time window:", nrow(obs_time_check), "\n")

# Write everything as dataframe and store

In [ ]:
wide_tables_all <- list(
  cond_era_all_wide   = cond_era_all_wide,
  drug_era_all_wide   = drug_era_all_wide,
  proc_all_wide       = proc_all_wide,
  obs_801_wide        = obs_801_wide,
  meas_706_wide       = meas_706_wide,
  age_continuous_wide = age_continuous_wide
)

sapply(wide_tables_all, function(x) class(x$person_id))

binary_tables <- list(
  cond_era_all_wide   = cond_era_all_wide,
  drug_era_all_wide   = drug_era_all_wide,
  proc_all_wide       = proc_all_wide,
  obs_801_wide        = obs_801_wide
)

continuous_tables <- list(
  meas_706_wide       = meas_706_wide,
  age_continuous_wide = age_continuous_wide
)

feature_matrix_wide <- data.frame(person_id = as.character(cohort_ids_int))

for (tbl in c(binary_tables, continuous_tables)) {
  feature_matrix_wide <- feature_matrix_wide %>% left_join(tbl, by = "person_id")
}

## 4. Binary features: NA -> 0. Continuous features: keep NA (missingness signal preserved)
binary_cols <- setdiff(unlist(lapply(binary_tables, names)), "person_id")
binary_cols <- intersect(binary_cols, names(feature_matrix_wide))
feature_matrix_wide[, binary_cols][is.na(feature_matrix_wide[, binary_cols])] <- 0L

## 5. Diagnostics
print(dim(feature_matrix_wide))
cat("Patients:          ", nrow(feature_matrix_wide), "\n")
cat("Features:          ", ncol(feature_matrix_wide) - 1, "\n")
cat("Expected patients: ", length(cohort_ids_int), "\n")
cat("Actual patients:   ", nrow(feature_matrix_wide), "\n")
cat("Duplicate IDs:     ", sum(duplicated(feature_matrix_wide$person_id)), "\n")

# per-table "did this join actually contribute" check -- catches a silently-failed join
for (nm in names(wide_tables_all)) {
  cols <- setdiff(names(wide_tables_all[[nm]]), "person_id")
  cols <- intersect(cols, names(feature_matrix_wide))
  if (length(cols) == 0) {
    cat(nm, ": no columns present in final matrix\n")
  } else {
    contributed <- sum(rowSums(!is.na(feature_matrix_wide[, cols, drop = FALSE])) > 0)
    cat(nm, ": contributed non-NA to", contributed, "patients (", length(cols), "cols)\n")
  }
}

cat("Meas NA rate: ", mean(is.na(feature_matrix_wide[, setdiff(names(meas_706_wide), "person_id")])), "\n")

In [ ]:
write.csv(feature_matrix_wide, "feature_matrix_wide_deaths.csv", row.names = FALSE)